# Day 5 — Fine-Tuned Models Evaluation & Synthetic QA Generation (v2 Prep)

Welcome to the **Day 5 Evaluation and Data Expansion Notebook**! In this session, we will:
1. Load both the **base** and **fine-tuned** (v1) versions of Qwen and Llama.
2. Run predictions on our unseen `test.json` dataset and calculate NLP evaluation metrics (BLEU, ROUGE-1, ROUGE-2, ROUGE-L).
3. Compare base vs. fine-tuned performances quantitatively and qualitatively.
4. **Generate 500+ synthetic QA pairs** focusing on weak/expansion areas to improve process operations and retail edge cases (Jira **KAN-32**).
5. Compile the new **`train_v2.json`** dataset and write configuration files for the **v2 training runs**.

---  
## Step 1: Mount Google Drive & Install Required Packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes wandb trl rouge-score nltk pandas matplotlib

---  
## Step 2: Initialize Workspace & Sync Files

In [ ]:
import os

project_dir = "/content/Retail"
gdrive_dir = None
for candidate in ["/content/drive/MyDrive/Retail LLM", "/content/drive/MyDrive/Retail"]:
    if os.path.isdir(candidate):
        gdrive_dir = candidate
        break
if gdrive_dir is None:
    gdrive_dir = "/content/drive/MyDrive/Retail LLM"
    print(f"[!] Project folder not found on Drive. Defaulting target back to: {gdrive_dir}")
else:
    print(f"[+] Detected active Drive folder: {gdrive_dir}")

os.makedirs(project_dir, exist_ok=True)
print("[*] Syncing scripts and data from Google Drive...")
!rsync -av --progress "{gdrive_dir}/" /content/Retail/
print("[+] Workspace sync complete.")

---  
## Step 3: Write Latest Evaluation & Generation Scripts to Workspace

In [ ]:
evaluate_code = "import os\nimport argparse\nimport json\nimport torch\nimport random\nfrom datasets import load_dataset\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\nfrom peft import PeftModel\nfrom rouge_score import rouge_scorer\nimport nltk\nfrom nltk.translate.bleu_score import sentence_bleu, SmoothingFunction\n\n# Download NLTK data if not present (handled quietly)\ntry:\n    nltk.data.find('tokenizers/punkt')\nexcept LookupError:\n    nltk.download('punkt', quiet=True)\ntry:\n    nltk.data.find('tokenizers/punkt_tab')\nexcept LookupError:\n    nltk.download('punkt_tab', quiet=True)\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=\"Evaluate fine-tuned model against reference dataset.\")\n    parser.add_argument(\n        \"--model_id\",\n        type=str,\n        required=True,\n        help=\"Hugging Face base model identifier (e.g. Qwen/Qwen2.5-7B-Instruct or meta-llama/Meta-Llama-3-8B-Instruct)\"\n    )\n    parser.add_argument(\n        \"--adapter_dir\",\n        type=str,\n        default=None,\n        help=\"Path to the LoRA adapter directory. If None, evaluates the base model only.\"\n    )\n    parser.add_argument(\n        \"--test_file\",\n        type=str,\n        default=\"data/processed/test.json\",\n        help=\"Path to the test JSON file.\"\n    )\n    parser.add_argument(\n        \"--output_file\",\n        type=str,\n        required=True,\n        help=\"Path to save the evaluation results JSON file.\"\n    )\n    parser.add_argument(\n        \"--num_samples\",\n        type=int,\n        default=100,\n        help=\"Number of random samples to evaluate (default: 100).\"\n    )\n    parser.add_argument(\n        \"--seed\",\n        type=int,\n        default=42,\n        help=\"Random seed for reproducibility.\"\n    )\n    return parser.parse_args()\n\ndef main():\n    args = parse_args()\n    random.seed(args.seed)\n    \n    print(\"\\n==============================================\")\n    print(f\"[*] Base Model: {args.model_id}\")\n    print(f\"[*] Adapter Path: {args.adapter_dir}\")\n    print(f\"[*] Output Path: {args.output_file}\")\n    print(\"==============================================\\n\")\n    \n    # 1. Load Tokenizer\n    print(\"[*] Loading tokenizer...\")\n    tokenizer = AutoTokenizer.from_pretrained(args.model_id, trust_remote_code=True)\n    if tokenizer.pad_token is None:\n        tokenizer.pad_token = tokenizer.eos_token\n        \n    # 2. Load Model in 4-bit Quantization (to fit in T4 GPU VRAM)\n    print(\"[*] Loading base model in 4-bit quantization...\")\n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_use_double_quant=True,\n        bnb_4bit_quant_type=\"nf4\",\n        bnb_4bit_compute_dtype=torch.float16\n    )\n    \n    base_model = AutoModelForCausalLM.from_pretrained(\n        args.model_id,\n        quantization_config=bnb_config,\n        device_map=\"auto\",\n        trust_remote_code=True,\n        torch_dtype=torch.float16\n    )\n    \n    # Force all bfloat16 parameters and buffers in the base model to float16 to prevent bfloat16 propagation\n    for name, param in base_model.named_parameters():\n        if param.dtype == torch.bfloat16:\n            param.data = param.data.to(torch.float16)\n    for name, buf in base_model.named_buffers():\n        if buf.dtype == torch.bfloat16:\n            buf.data = buf.data.to(torch.float16)\n            \n    # 3. Load LoRA Adapter if provided\n    if args.adapter_dir:\n        print(f\"[*] Loading LoRA adapter from {args.adapter_dir}...\")\n        model = PeftModel.from_pretrained(base_model, args.adapter_dir)\n    else:\n        print(\"[*] No adapter provided. Evaluating raw base model.\")\n        model = base_model\n        \n    model.eval()\n    \n    # 4. Load Test Dataset\n    print(f\"[*] Loading test file: {args.test_file}...\")\n    if not os.path.exists(args.test_file):\n        raise FileNotFoundError(f\"Test file not found: {args.test_file}\")\n        \n    with open(args.test_file, \"r\", encoding=\"utf-8\") as f:\n        test_data = json.load(f)\n        \n    if len(test_data) > args.num_samples:\n        print(f\"[*] Sampling {args.num_samples} records from {len(test_data)} total test records.\")\n        # Ensure repeatable sampling using seeded random\n        test_samples = random.sample(test_data, args.num_samples)\n    else:\n        print(f\"[*] Using all {len(test_data)} test records.\")\n        test_samples = test_data\n        \n    # 5. Setup Scorers\n    rouge_scorer_inst = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)\n    smoothing = SmoothingFunction().method1\n    \n    results = []\n    total_r1, total_r2, total_rl, total_bleu = 0.0, 0.0, 0.0, 0.0\n    \n    # 6. Evaluation Generation Loop\n    print(\"\\n[*] Starting text generation and evaluation...\")\n    for idx, sample in enumerate(test_samples):\n        instruction = sample[\"instruction\"]\n        reference = sample[\"response\"]\n        \n        # Build prompt using SFT instruction-tuning prompt template\n        prompt = f\"Below is an instruction that describes a task. Write a response that appropriately completes the request.\\n\\n### Instruction:\\n{instruction}\\n\\n### Response:\\n\"\n        \n        inputs = tokenizer(prompt, return_tensors=\"pt\").to(\"cuda\")\n        \n        with torch.no_grad():\n            outputs = model.generate(\n                **inputs,\n                max_new_tokens=150,\n                temperature=0.7,\n                top_p=0.9,\n                do_sample=True,\n                pad_token_id=tokenizer.eos_token_id\n            )\n            \n        # Slice outputs to retrieve only the generated completion (ignoring prompt tokens)\n        prompt_len = inputs.input_ids.shape[1]\n        generation_tokens = outputs[0][prompt_len:]\n        prediction = tokenizer.decode(generation_tokens, skip_special_tokens=True).strip()\n        \n        # Compute ROUGE\n        rouge_scores = rouge_scorer_inst.score(reference, prediction)\n        r1 = rouge_scores['rouge1'].fmeasure\n        r2 = rouge_scores['rouge2'].fmeasure\n        rl = rouge_scores['rougeL'].fmeasure\n        \n        # Compute BLEU (word level)\n        ref_tokens = nltk.word_tokenize(reference.lower())\n        pred_tokens = nltk.word_tokenize(prediction.lower())\n        bleu = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothing)\n        \n        # Accumulate scores\n        total_r1 += r1\n        total_r2 += r2\n        total_rl += rl\n        total_bleu += bleu\n        \n        results.append({\n            \"instruction\": instruction,\n            \"reference\": reference,\n            \"prediction\": prediction,\n            \"metrics\": {\n                \"rouge1\": r1,\n                \"rouge2\": r2,\n                \"rougeL\": rl,\n                \"bleu\": bleu,\n                \"length\": len(prediction)\n            }\n        })\n        \n        if (idx + 1) % 10 == 0 or (idx + 1) == len(test_samples):\n            print(f\"    Processed {idx + 1}/{len(test_samples)} samples...\")\n            \n    # Calculate Summary Scores\n    num_evaluated = len(test_samples)\n    summary = {\n        \"mean_rouge1\": total_r1 / num_evaluated,\n        \"mean_rouge2\": total_r2 / num_evaluated,\n        \"mean_rougeL\": total_rl / num_evaluated,\n        \"mean_bleu\": total_bleu / num_evaluated\n    }\n    \n    output_data = {\n        \"model_id\": args.model_id,\n        \"adapter_dir\": args.adapter_dir,\n        \"summary\": summary,\n        \"results\": results\n    }\n    \n    # 7. Write Results\n    os.makedirs(os.path.dirname(args.output_file), exist_ok=True)\n    with open(args.output_file, \"w\", encoding=\"utf-8\") as f:\n        json.dump(output_data, f, ensure_ascii=False, indent=2)\n        \n    print(\"\\n========================= SUMMARY =========================\")\n    print(f\"[+] ROUGE-1 F-Measure: {summary['mean_rouge1']:.4f}\")\n    print(f\"[+] ROUGE-2 F-Measure: {summary['mean_rouge2']:.4f}\")\n    print(f\"[+] ROUGE-L F-Measure: {summary['mean_rougeL']:.4f}\")\n    print(f\"[+] BLEU Score:        {summary['mean_bleu']:.4f}\")\n    print(\"===========================================================\\n\")\n    print(f\"[+] Detailed evaluation records saved to: {args.output_file}\")\n\nif __name__ == \"__main__\":\n    main()\n";
with open("/content/Retail/src/evaluate.py", "w", encoding="utf-8") as f:
    f.write(evaluate_code)
print("[+] src/evaluate.py successfully written.")

In [ ]:
generate_code = "import os\nimport json\nimport random\n\ndef generate_retail_qa():\n    qa_list = []\n    \n    # Define vocabulary and templates for retail issues\n    order_ids = [f\"ORD-{random.randint(10000, 99999)}\" for _ in range(100)]\n    tracking_statuses = [\"in transit\", \"delayed at sorting facility\", \"customs clearance delay\", \"out for delivery\"]\n    refund_reasons = [\"damaged during shipping\", \"wrong size delivered\", \"defective item\", \"not as described\"]\n    payment_methods = [\"Credit Card\", \"PayPal\", \"Apple Pay\", \"Google Pay\"]\n    carrier_names = [\"FedEx\", \"DHL\", \"UPS\", \"USPS\"]\n    \n    # 1. Shipping & Transit Delays\n    for i in range(70):\n        oid = random.choice(order_ids)\n        status = random.choice(tracking_statuses)\n        carrier = random.choice(carrier_names)\n        \n        q_templates = [\n            f\"My order {oid} is taking too long. Can you track it?\",\n            f\"Why is my package {oid} delayed? It has been in transit for days.\",\n            f\"I checked my tracking for {oid} and it says '{status}' via {carrier}. What does this mean?\",\n            f\"Where is my package {oid}? The carrier is {carrier} and I haven't received it.\",\n            f\"Can you get an update on order {oid}? It says '{status}'.\"\n        ]\n        \n        a_templates = [\n            f\"I sincerely apologize for the delay with your order {oid}. Looking at the tracking data via {carrier}, the shipment is currently {status}. We are contacting the carrier to expedite delivery and will keep you updated.\",\n            f\"We are sorry for the inconvenience. Your package {oid} is currently experiencing a {status}. This is usually resolved within 48 hours. We are monitoring it closely for you.\",\n            f\"Thank you for contacting customer support. Order {oid} has a status of '{status}' at the {carrier} facility. We expect it to resume transit by tomorrow. Please let us know if it does not arrive by Friday.\"\n        ]\n        \n        qa_list.append({\n            \"instruction\": random.choice(q_templates),\n            \"response\": random.choice(a_templates),\n            \"category\": \"shipping_delay\",\n            \"intent\": \"track_order\"\n        })\n        \n    # 2. Refund Disputes & Damage\n    for i in range(70):\n        oid = random.choice(order_ids)\n        reason = random.choice(refund_reasons)\n        method = random.choice(payment_methods)\n        \n        q_templates = [\n            f\"I received my order {oid} but the item is {reason}. I want a full refund.\",\n            f\"Can I get a refund for {oid}? The product was {reason}.\",\n            f\"My order {oid} was delivered but it's {reason}. How do I return it and get my money back?\",\n            f\"I want to request a return on {oid} because of a {reason}.\",\n            f\"The package {oid} arrived damaged. Can you process a refund back to my {method}?\"\n        ]\n        \n        a_templates = [\n            f\"I am very sorry to hear that your order {oid} arrived {reason}. We will gladly process a full refund to your original payment method ({method}). I have also emailed you a prepaid shipping label so you can return the item.\",\n            f\"We apologize for the defective item in order {oid}. A return request has been authorized for the reason: '{reason}'. Your refund of the purchase amount will be processed back to your {method} as soon as we scan the package.\",\n            f\"That is certainly not the experience we want you to have. I have initiated a refund for order {oid} because it was {reason}. The funds should appear on your {method} statement within 3-5 business days.\"\n        ]\n        \n        qa_list.append({\n            \"instruction\": random.choice(q_templates),\n            \"response\": random.choice(a_templates),\n            \"category\": \"refund_request\",\n            \"intent\": \"dispute_refund\"\n        })\n        \n    # 3. Order Cancellations & Modifications\n    for i in range(60):\n        oid = random.choice(order_ids)\n        \n        q_templates = [\n            f\"I need to cancel my order {oid} immediately before it ships.\",\n            f\"Can I change the shipping address on my order {oid}?\",\n            f\"I accidentally ordered the wrong size for order {oid}. Can we update it?\",\n            f\"Can you stop order {oid}? I changed my mind.\",\n            f\"Please cancel order {oid} and refund my card.\"\n        ]\n        \n        a_templates = [\n            f\"I have checked order {oid} and since it has not left our warehouse yet, I was able to successfully cancel it. A confirmation email has been sent and a full refund is being processed.\",\n            f\"I have successfully updated the shipping address for your order {oid} as requested. Please check your email for the updated order confirmation details.\",\n            f\"We have updated the item details for order {oid} in our system. You will receive a new confirmation email shortly reflecting these changes.\"\n        ]\n        \n        qa_list.append({\n            \"instruction\": random.choice(q_templates),\n            \"response\": random.choice(a_templates),\n            \"category\": \"order_modification\",\n            \"intent\": \"cancel_order\"\n        })\n        \n    # 4. Custom Orders & Requests\n    for i in range(60):\n        oid = random.choice(order_ids)\n        \n        q_templates = [\n            f\"Do you accept custom branding on orders over 100 units?\",\n            f\"Can I add engraving or customization to my order {oid}?\",\n            f\"What is the lead time for customized bulk retail shipments?\",\n            f\"Is it possible to request gift wrapping and a custom note for order {oid}?\"\n        ]\n        \n        a_templates = [\n            f\"Yes, we offer custom branding options (including logo printing) for corporate and bulk orders over 100 units. Please email our sales team at bulk@retailcorp.com with your specifications.\",\n            f\"Certainly! I have added your requested engraving details to order {oid}. This customization adds approximately 24 hours of processing time before shipping.\",\n            f\"For bulk customized shipments, the standard production lead time is 10-14 business days after logo approval, followed by 3-5 days of transit time via FedEx.\"\n        ]\n        \n        qa_list.append({\n            \"instruction\": random.choice(q_templates),\n            \"response\": random.choice(a_templates),\n            \"category\": \"customization\",\n            \"intent\": \"custom_request\"\n        })\n        \n    return qa_list\n\ndef generate_manufacturing_qa():\n    qa_list = []\n    \n    # Define vocabulary and templates for manufacturing processes\n    dmaic_phases = [\"Define\", \"Measure\", \"Analyze\", \"Improve\", \"Control\"]\n    spc_charts = [\"X-bar chart\", \"R-chart\", \"p-chart\", \"c-chart\"]\n    wastes = [\"Overproduction\", \"Waiting\", \"Transport\", \"Overprocessing\", \"Inventory\", \"Motion\", \"Defects\"]\n    control_actions = [\"quarantine the batch\", \"re-calibrate the machine sensors\", \"perform a root cause analysis\", \"halt the assembly line\"]\n    \n    # 1. Lean Six Sigma & DMAIC Operations\n    for i in range(100):\n        phase = random.choice(dmaic_phases)\n        waste = random.choice(wastes)\n        \n        q_templates = [\n            f\"What is the main goal of the {phase} phase in a Lean Six Sigma DMAIC project?\",\n            f\"How do we identify and eliminate '{waste}' waste in manufacturing assembly lines?\",\n            f\"Which tools are commonly used during the {phase} stage of process operations?\",\n            f\"Can you explain how eliminating the '{waste}' waste improves our Lean Six Sigma score?\"\n        ]\n        \n        a_templates = [\n            f\"The primary goal of the {phase} phase is to establish a clear structure for process improvement. In operations, this involves mapping inputs/outputs, verifying measurement systems, and stabilizing variation. For instance, focusing on reducing '{waste}' directly optimizes throughput.\",\n            f\"To systematically address '{waste}' waste, we utilize Value Stream Mapping (VSM) and Standardized Work instructions. During the {phase} phase, we gather cycle-time data to isolate non-value-added activities and implement 5S practices to stabilize the manufacturing floor.\"\n        ]\n        \n        qa_list.append({\n            \"instruction\": random.choice(q_templates),\n            \"response\": random.choice(a_templates),\n            \"domain\": \"lean_six_sigma\"\n        })\n        \n    # 2. Quality Control & SPC charts\n    for i in range(100):\n        chart = random.choice(spc_charts)\n        action = random.choice(control_actions)\n        \n        q_templates = [\n            f\"What should the floor supervisor do if an alert points to an out-of-control point on the {chart}?\",\n            f\"How does an operator determine if variation on the {chart} is due to common cause or assignable cause?\",\n            f\"What standard operating procedure is triggered when the {chart} violates Western Electric rules?\",\n            f\"How often should we calibrate our sensors to prevent false alarms on the {chart}?\"\n        ]\n        \n        a_templates = [\n            f\"When a data point falls outside the control limits on the {chart}, the supervisor must immediately {action}. An assignable cause variation must be investigated using a Fishbone diagram and process logs.\",\n            f\"An out-of-control signal on the {chart} implies assignable cause variation. The operator is trained to {action} immediately and document the corrective action in the SPC quality logbook.\"\n        ]\n        \n        qa_list.append({\n            \"instruction\": random.choice(q_templates),\n            \"response\": random.choice(a_templates),\n            \"domain\": \"statistical_process_control\"\n        })\n        \n    # 3. Root Cause Analysis (RCA) & Troubleshooting\n    for i in range(60):\n        action = random.choice(control_actions)\n        \n        q_templates = [\n            \"How do we apply the '5 Whys' methodology to isolate a machinery defect?\",\n            \"What is the difference between corrective action and preventive action (CAPA) in quality audits?\",\n            f\"If a component tolerance check fails, what is the sequence of troubleshooting steps to take?\",\n            \"How do we construct a Pareto chart to prioritize quality defect mitigation?\"\n        ]\n        \n        a_templates = [\n            f\"The '5 Whys' begins with the defect symptom and drills down to the mechanical or procedural root cause. Once the root cause is isolated, we implement a permanent corrective action, such as to {action}.\",\n            f\"Corrective action addresses an existing process failure, requiring the line team to {action}. Preventive action addresses systemic issues to stop future failure modes before they occur.\"\n        ]\n        \n        qa_list.append({\n            \"instruction\": random.choice(q_templates),\n            \"response\": random.choice(a_templates),\n            \"domain\": \"root_cause_analysis\"\n        })\n        \n    return qa_list\n\ndef main():\n    print(\"[*] Generating synthetic domain-specific QA expansion pairs...\")\n    \n    # Determine project root dynamically relative to the script location\n    script_dir = os.path.dirname(os.path.abspath(__file__))\n    project_root = os.path.dirname(script_dir)\n    print(f\"[*] Detected project root directory: {project_root}\")\n    \n    retail_synthetic = generate_retail_qa()\n    mfg_synthetic = generate_manufacturing_qa()\n    \n    all_synthetic = retail_synthetic + mfg_synthetic\n    print(f\"[+] Generated {len(retail_synthetic)} Retail/Ecom samples.\")\n    print(f\"[+] Generated {len(mfg_synthetic)} Manufacturing samples.\")\n    print(f\"[+] Total Synthetic QA pairs: {len(all_synthetic)}\")\n    \n    # Save synthetic records\n    synthetic_file = os.path.join(project_root, \"data/processed/synthetic_qa.json\")\n    os.makedirs(os.path.dirname(synthetic_file), exist_ok=True)\n    with open(synthetic_file, \"w\", encoding=\"utf-8\") as f:\n        json.dump(all_synthetic, f, ensure_ascii=False, indent=2)\n    print(f\"[+] Synthetic data saved to {synthetic_file}\")\n    \n    # Merge with original train.json\n    train_file = os.path.join(project_root, \"data/processed/train.json\")\n    train_v2_file = os.path.join(project_root, \"data/processed/train_v2.json\")\n    \n    if os.path.exists(train_file):\n        print(f\"[*] Loading original train dataset from {train_file}...\")\n        with open(train_file, \"r\", encoding=\"utf-8\") as f:\n            original_train = json.load(f)\n            \n        print(f\"[*] Merging {len(original_train)} original records with {len(all_synthetic)} synthetic records...\")\n        train_v2 = original_train + all_synthetic\n        \n        # Shuffle train_v2 for model training distribution\n        random.shuffle(train_v2)\n        \n        with open(train_v2_file, \"w\", encoding=\"utf-8\") as f:\n            json.dump(train_v2, f, ensure_ascii=False, indent=2)\n            \n        print(f\"[+] train_v2.json successfully compiled and saved to {train_v2_file}\")\n        print(f\"    - Total train_v2 records: {len(train_v2)}\")\n    else:\n        print(f\"[!] Warning: Original train file {train_file} not found. Cannot merge. Creating standalone train_v2.json.\")\n        with open(train_v2_file, \"w\", encoding=\"utf-8\") as f:\n            json.dump(all_synthetic, f, ensure_ascii=False, indent=2)\n            \n    # Write configs/qwen_lora_config_v2.json &configs/llama_lora_config_v2.json\n    print(\"[*] Creating configs for v2 fine-tuning runs...\")\n    \n    qwen_v2_cfg = {\n        \"model_type\": \"qwen\",\n        \"model_id\": \"Qwen/Qwen2.5-7B-Instruct\",\n        \"output_dir\": \"models/qwen_v2\",\n        \"peft_settings\": {\n            \"r\": 16,\n            \"lora_alpha\": 32,\n            \"target_modules\": [\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\", \"gate_proj\", \"up_proj\", \"down_proj\"],\n            \"lora_dropout\": 0.05,\n            \"bias\": \"none\"\n        },\n        \"bnb_4bit_compute_dtype\": \"float16\"\n    }\n    \n    llama_v2_cfg = {\n        \"model_type\": \"llama\",\n        \"model_id\": \"meta-llama/Meta-Llama-3-8B-Instruct\",\n        \"output_dir\": \"models/llama_v2\",\n        \"peft_settings\": {\n            \"r\": 16,\n            \"lora_alpha\": 32,\n            \"target_modules\": [\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\", \"gate_proj\", \"up_proj\", \"down_proj\"],\n            \"lora_dropout\": 0.05,\n            \"bias\": \"none\"\n        },\n        \"bnb_4bit_compute_dtype\": \"float16\"\n    }\n    \n    qwen_cfg_path = os.path.join(project_root, \"configs/qwen_lora_config_v2.json\")\n    llama_cfg_path = os.path.join(project_root, \"configs/llama_lora_config_v2.json\")\n    \n    with open(qwen_cfg_path, \"w\", encoding=\"utf-8\") as f:\n        json.dump(qwen_v2_cfg, f, indent=2)\n    with open(llama_cfg_path, \"w\", encoding=\"utf-8\") as f:\n        json.dump(llama_v2_cfg, f, indent=2)\n        \n    print(\"[+] v2 configuration files written successfully.\")\n\nif __name__ == \"__main__\":\n    main()\n";
with open("/content/Retail/src/generate_synthetic_data.py", "w", encoding="utf-8") as f:
    f.write(generate_code)
print("[+] src/generate_synthetic_data.py successfully written.")

---  
## Step 4: Evaluate Qwen Base vs. Fine-Tuned (Capped at 100 Samples)

In [ ]:
# 1. Run evaluation on Base Qwen Model
!python /content/Retail/src/evaluate.py \
    --model_id Qwen/Qwen2.5-7B-Instruct \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/qwen_base_results.json \
    --num_samples 100

In [ ]:
# 2. Run evaluation on Fine-Tuned Qwen Model
!python /content/Retail/src/evaluate.py \
    --model_id Qwen/Qwen2.5-7B-Instruct \
    --adapter_dir "{gdrive_dir}/models/qwen_v1" \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/qwen_finetuned_results.json \
    --num_samples 100

---  
## Step 5: Evaluate Llama Base vs. Fine-Tuned (Capped at 100 Samples)

In [ ]:
from google.colab import userdata
import os

if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
        print("[+] Successfully loaded HF_TOKEN from Colab Secrets.")
    except Exception:
        print("[-] Warning: HF_TOKEN secret not found.")

# 1. Run evaluation on Base Llama Model
!python /content/Retail/src/evaluate.py \
    --model_id meta-llama/Meta-Llama-3-8B-Instruct \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/llama_base_results.json \
    --num_samples 100

In [ ]:
# 2. Run evaluation on Fine-Tuned Llama Model
!python /content/Retail/src/evaluate.py \
    --model_id meta-llama/Meta-Llama-3-8B-Instruct \
    --adapter_dir "{gdrive_dir}/models/llama_v1" \
    --test_file /content/Retail/data/processed/test.json \
    --output_file /content/Retail/models/evaluation/llama_finetuned_results.json \
    --num_samples 100

---  
## Step 6: Plot Comparative Results & Display Side-by-Side QA

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import os

eval_dir = "/content/Retail/models/evaluation"
models = {
    "Qwen Base": "qwen_base_results.json",
    "Qwen Fine-Tuned": "qwen_finetuned_results.json",
    "Llama Base": "llama_base_results.json",
    "Llama Fine-Tuned": "llama_finetuned_results.json"
}

metrics_summary = {}
for name, fname in models.items():
    fpath = os.path.join(eval_dir, fname)
    if os.path.exists(fpath):
        with open(fpath, "r") as f:
            data = json.load(f)
            metrics_summary[name] = data["summary"]
    else:
        print(f"[!] Warning: {fname} results file not found.")

if metrics_summary:
    df = pd.DataFrame(metrics_summary).T
    print("\n===================== OVERALL SCORES =====================")
    print(df.round(4))
    print("==========================================================\n")
    
    # Plot comparative chart
    fig, ax = plt.subplots(figsize=(10, 6))
    df[["mean_rouge1", "mean_rougeL", "mean_bleu"]].plot(kind="bar", ax=ax)
    ax.set_title("Base Model vs. Fine-Tuned Model Performance Comparison")
    ax.set_ylabel("Score (Higher is Better)")
    ax.set_xticklabels(df.index, rotation=15)
    ax.legend(["ROUGE-1", "ROUGE-L", "BLEU"])
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("[X] No evaluation metrics could be loaded.")

In [ ]:
# Show side-by-side responses for interactive validation
import pandas as pd
import random

qwen_ft_path = os.path.join(eval_dir, "qwen_finetuned_results.json")
qwen_base_path = os.path.join(eval_dir, "qwen_base_results.json")

if os.path.exists(qwen_ft_path) and os.path.exists(qwen_base_path):
    with open(qwen_ft_path) as f: ft_data = json.load(f)["results"]
    with open(qwen_base_path) as f: base_data = json.load(f)["results"]
    
    compare_list = []
    for i in range(min(5, len(ft_data))):  # Display 5 sample predictions
        compare_list.append({
            "Instruction": ft_data[i]["instruction"],
            "Reference Answer": ft_data[i]["reference"],
            "Base Model Response": base_data[i]["prediction"],
            "Fine-Tuned Response": ft_data[i]["prediction"]
        })
    
    df_compare = pd.DataFrame(compare_list)
    pd.set_option('display.max_colwidth', None)
    display(df_compare.style.set_properties(**{'text-align': 'left'}))
else:
    print("[!] Qwen result files not found to build comparison table.")

---  
## Step 7: Generate Synthetic QA Expansion & Compile train_v2.json (Jira KAN-32)

In [ ]:
# Execute synthetic dataset generator
!python /content/Retail/src/generate_synthetic_data.py

In [ ]:
# Verify v2 dataset details
import json
import os

train_v2_path = "/content/Retail/data/processed/train_v2.json"
if os.path.exists(train_v2_path):
    with open(train_v2_path, "r") as f:
        data_v2 = json.load(f)
    print(f"[+] Verification Successful!")
    print(f"    - train_v2.json records: {len(data_v2)}")
    print(f"    - Sample instruction: {data_v2[-1]['instruction']}")
    print(f"    - Sample response: {data_v2[-1]['response']}")
else:
    print("[X] Error: train_v2.json not generated successfully.")

---  
## Step 8: Sync Results, train_v2.json, and v2 Configs back to Google Drive

In [ ]:
print(f"[*] Saving evaluation reports, train_v2.json, and v2 configs to Drive: {gdrive_dir}...")
local_eval_path = "/content/Retail/models/evaluation"
drive_eval_path = os.path.join(gdrive_dir, "models", "evaluation")
os.makedirs(drive_eval_path, exist_ok=True)
!cp -v "{local_eval_path}/"*.json "{drive_eval_path}/"

# Sync processed v2 splits and configs
drive_processed_path = os.path.join(gdrive_dir, "data", "processed")
os.makedirs(drive_processed_path, exist_ok=True)
!cp -v /content/Retail/data/processed/train_v2.json "{drive_processed_path}/"
!cp -v /content/Retail/data/processed/synthetic_qa.json "{drive_processed_path}/"

drive_configs_path = os.path.join(gdrive_dir, "configs")
os.makedirs(drive_configs_path, exist_ok=True)
!cp -v /content/Retail/configs/*_v2.json "{drive_configs_path}/"

print("[+] Backup complete.")